# Complete retrospective Stage-2: strongest model first

Unexecuted operator notebook for the complete R-only pipeline. It invokes the reviewed production CLIs for Variant A, streamed factored-bank construction/audit, full-model sanity, strongest-first training, backward ablations, and final retrospective evaluation. It never executes prospective data and never enables descriptor coupling. There are no review gates between internal stages; the single long-run authorization flag is in the training cell.


In [ ]:
from pathlib import Path, PureWindowsPath
from collections import Counter
import copy, hashlib, importlib, json, os, re, subprocess, sys, tomllib

REPOSITORY_URL = 'https://github.com/GuillermoTafoya/MRIxFields.git'
BASE_COMMIT = '09605a0ce5a3c14d3e19ea7c719405d5cc5d35b3'
IMPLEMENTATION_REF = input('Reviewed unified implementation commit SHA: ').strip()
if re.fullmatch(r'[0-9a-f]{40}', IMPLEMENTATION_REF) is None:
    raise ValueError('Pin the exact externally reviewed 40-character implementation SHA.')
REPO_DIR = Path('/content/MRIxFields-stage2-unified')
if REPO_DIR.exists():
    raise FileExistsError('Start a fresh runtime; refusing to mutate an existing checkout.')
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin', IMPLEMENTATION_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', IMPLEMENTATION_REF], cwd=REPO_DIR, check=True)
def git_text(*args):
    return subprocess.check_output(['git', *args], cwd=REPO_DIR, text=True).strip()
if git_text('rev-parse', 'HEAD') != IMPLEMENTATION_REF:
    raise RuntimeError('Detached implementation SHA mismatch.')
if subprocess.run(['git', 'merge-base', '--is-ancestor', BASE_COMMIT, 'HEAD'], cwd=REPO_DIR).returncode:
    raise RuntimeError('Unified implementation is not based on the reviewed merged main.')
if git_text('status', '--porcelain') or subprocess.run(['git', 'symbolic-ref', '-q', 'HEAD'], cwd=REPO_DIR, capture_output=True).returncode == 0:
    raise RuntimeError('Checkout must be detached and clean.')
print({'head': IMPLEMENTATION_REF, 'base': BASE_COMMIT, 'clean': True, 'detached': True})


In [ ]:
# Install only dependencies declared by this detached checkout and force its imports.
with (REPO_DIR / 'pyproject.toml').open('rb') as handle:
    project = tomllib.load(handle)['project']
optional = project['optional-dependencies']
requirements = list(dict.fromkeys([*project['dependencies'], *optional['evaluation'], *optional['official-evaluation']]))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *requirements], check=True)
SOURCE_DIR = str(REPO_DIR / 'src')
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['PYTHONPATH'] = SOURCE_DIR + os.pathsep + os.environ.get('PYTHONPATH', '')
sys.path.insert(0, SOURCE_DIR)
for name in tuple(sys.modules):
    if name == 'fieldbridge' or name.startswith('fieldbridge.'):
        del sys.modules[name]
importlib.invalidate_caches()
import fieldbridge
if REPO_DIR not in Path(fieldbridge.__file__).resolve().parents:
    raise RuntimeError('FieldBridge import escaped the detached checkout.')
CLI_ENV = os.environ.copy()
CLI_ENV['PYTHONPATH'] = SOURCE_DIR + os.pathsep + CLI_ENV.get('PYTHONPATH', '')
if git_text('status', '--porcelain'):
    raise RuntimeError('Dependency installation changed the checkout.')
print({'fieldbridge_import': str(Path(fieldbridge.__file__).resolve()), 'requirements': requirements})


In [ ]:
# All operator paths are supplied once, before any scientific array is loaded.
from google.colab import drive
drive.mount('/content/drive')
FROZEN_SPLIT_V3_JSON = Path(input('Immutable split-v3 JSON: ').strip()).expanduser()
RETROSPECTIVE_DATA_ROOT = Path(input('Retrospective data root: ').strip()).expanduser()
FROZEN_STAGE1_VAE_CONFIG = Path(input('Frozen Stage-1 VAE config: ').strip()).expanduser()
FROZEN_VAE_CHECKPOINT = Path(input('Frozen VAE checkpoint: ').strip()).expanduser()
GATE01_RESULT_JSON = Path(input('Reviewed Gate 0.1 result JSON: ').strip()).expanduser()
PAIRED_R_VALIDATION_MANIFEST = Path(input('Complete paired R/validation manifest with Stage-1 ceilings: ').strip()).expanduser()
BASELINE_PREDICTIONS_MANIFEST = Path(input('Sealed calibrated-identity/original-SB-v2 prediction manifest: ').strip()).expanduser()
EXTERNAL_OUTPUT_ROOT = Path(input('New or exactly resumable external output root: ').strip()).expanduser()
EXPECTED_VAE_CONFIG_SHA256 = input('Frozen VAE config SHA-256: ').strip().lower()
EXPECTED_VAE_CHECKPOINT_SHA256 = input('Frozen VAE checkpoint SHA-256: ').strip().lower()
EXPECTED_PAIRED_MANIFEST_SHA256 = input('Paired R/validation manifest file SHA-256: ').strip().lower()
EXPECTED_BASELINE_PREDICTIONS_SHA256 = input('Baseline-predictions manifest file SHA-256: ').strip().lower()
EXPECTED_SPLIT_V3_SHA256 = 'f6a19d7a31c4c3bb73edd92088ea078192e88ee4b276309bad81c548ab7f94d5'
EXPECTED_RETROSPECTIVE_FIT_INVENTORY_SHA256 = 'cbe885f73a307065418ea80296d6cfd6d634edeb3281f503cf90e149800409e7'
FROZEN_RETROSPECTIVE_QUALIFICATION_INVENTORY_SHA256 = '569a17b1316a47a0c95c42c649f4aab61f8fe8c9cf7d0582c411f544c9b23173'
EXPECTED_GATE01_RESULT_SHA256 = '454747cd3e4b1376855915244a7c40fe281b758150e86f584fbea96f94d531f5'
REVIEWED_WINDOWS_SOURCE_ROOT = r'D:\MRI_Field_2026\Data'
for label, value in {'VAE config': EXPECTED_VAE_CONFIG_SHA256, 'VAE checkpoint': EXPECTED_VAE_CHECKPOINT_SHA256, 'paired manifest': EXPECTED_PAIRED_MANIFEST_SHA256, 'baseline predictions': EXPECTED_BASELINE_PREDICTIONS_SHA256}.items():
    if re.fullmatch(r'[0-9a-f]{64}', value) is None:
        raise ValueError(f'{label} requires an exact lowercase SHA-256.')


In [ ]:
# Deterministic split remap and byte-identical merged inventory arithmetic; no arrays open here.
from fieldbridge.data.manifests import record_from_mapping
from fieldbridge.data.photometry_factorization import all_photometry_domain_labels, assert_variant_a_external_path, sha256_file, sha256_json, sha256_text, write_json_atomic
from fieldbridge.data.vae_splits import VaeSplits, load_vae_splits, vae_splits_fingerprint, vae_splits_recovery_fingerprint_v3
import fieldbridge.cli as fb_cli
file_inputs = {
 'split': (FROZEN_SPLIT_V3_JSON, EXPECTED_SPLIT_V3_SHA256),
 'vae_config': (FROZEN_STAGE1_VAE_CONFIG, EXPECTED_VAE_CONFIG_SHA256),
 'vae_checkpoint': (FROZEN_VAE_CHECKPOINT, EXPECTED_VAE_CHECKPOINT_SHA256),
 'gate01': (GATE01_RESULT_JSON, EXPECTED_GATE01_RESULT_SHA256),
 'paired': (PAIRED_R_VALIDATION_MANIFEST, EXPECTED_PAIRED_MANIFEST_SHA256),
 'baselines': (BASELINE_PREDICTIONS_MANIFEST, EXPECTED_BASELINE_PREDICTIONS_SHA256),
}
for label, (path, expected) in file_inputs.items():
    path = assert_variant_a_external_path(path, repo_root=REPO_DIR)
    if not path.is_file() or sha256_file(path) != expected:
        raise RuntimeError(f'{label} missing or SHA-256 mismatch: {path}')
data_root = assert_variant_a_external_path(RETROSPECTIVE_DATA_ROOT, repo_root=REPO_DIR).resolve(strict=True)
output_root = assert_variant_a_external_path(EXTERNAL_OUTPUT_ROOT, repo_root=REPO_DIR)
if not data_root.is_dir(): raise NotADirectoryError(data_root)
output_root.mkdir(parents=True, exist_ok=True)
original_bytes = FROZEN_SPLIT_V3_JSON.read_bytes()
if hashlib.sha256(original_bytes).hexdigest() != EXPECTED_SPLIT_V3_SHA256:
    raise RuntimeError('Immutable split changed.')
original = load_vae_splits(FROZEN_SPLIT_V3_JSON)
original_membership = vae_splits_fingerprint(original)
original_recovery = vae_splits_recovery_fingerprint_v3(original)
payload = copy.deepcopy(json.loads(original_bytes.decode('utf-8')))
old_root = PureWindowsPath(REVIEWED_WINDOWS_SOURCE_ROOT)
def remap(raw):
    source = PureWindowsPath(str(raw)); lhs = tuple(x.casefold() for x in source.parts); rhs = tuple(x.casefold() for x in old_root.parts)
    if not source.is_absolute() or lhs[:len(rhs)] != rhs:
        raise ValueError(f'Path outside reviewed Windows root: {raw}')
    suffix = source.parts[len(old_root.parts):]
    if not suffix or any(x in {'', '.', '..'} for x in suffix): raise ValueError(f'Unsafe path: {raw}')
    mapped = data_root.joinpath(*suffix).resolve(strict=False)
    mapped.relative_to(data_root)
    return mapped
mapping = []
for split_name in ('train', 'validation', 'test'):
    for record in payload['splits'][split_name]:
        old = str(record['image_path']); new = remap(old); record['image_path'] = str(new)
        mapping.append({'split': split_name, 'record_identity': str(record.get('case_id', '')), 'old_path': old, 'new_path': str(new)})
mapping.sort(key=lambda x: (x['split'], x['record_identity'], x['old_path']))
mapping_sha = sha256_json(mapping)
metadata = dict(payload.get('metadata', {})); metadata['colab_path_remap'] = {'contract': 'stage2-colab-windows-root-remap-v1', 'original_split_file_sha256': EXPECTED_SPLIT_V3_SHA256, 'original_membership_fingerprint': original_membership, 'original_recovery_fingerprint_v3': original_recovery, 'reviewed_old_root': str(old_root), 'operational_new_root': str(data_root), 'remapped_record_count': len(mapping), 'mapping_identity_sha256': mapping_sha}; payload['metadata'] = metadata
def records(name): return tuple(record_from_mapping(x) for x in payload['splits'][name])
candidate = VaeSplits(train=records('train'), validation=records('validation'), test=records('test'), seed=int(payload['seed']), fractions=tuple(float(x) for x in payload['fractions']), metadata=metadata)
membership = vae_splits_fingerprint(candidate); recovery = vae_splits_recovery_fingerprint_v3(candidate)
if membership != original_membership: raise RuntimeError('Remap changed membership.')
assignments = lambda split: {name: tuple(sorted(r.case_id for r in split.records_for(name))) for name in ('train', 'validation', 'test')}
if assignments(candidate) != assignments(original): raise RuntimeError('Remap changed split assignments.')
payload['fingerprint'] = membership; payload['recovery_fingerprint_v3'] = recovery
OPERATIONAL_SPLIT = output_root / 'split_v3_colab_operational.json'
expected_bytes = (json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + '\n').encode()
if OPERATIONAL_SPLIT.exists():
    if OPERATIONAL_SPLIT.read_bytes() != expected_bytes: raise RuntimeError('Existing operational split is not exact resume.')
else: write_json_atomic(OPERATIONAL_SPLIT, payload)
splits = load_vae_splits(OPERATIONAL_SPLIT)
fit_records, fit_excluded = fb_cli._select_variant_a_retrospective_records(splits.train, split='train')
qualification_records, qualification_excluded = fb_cli._select_variant_a_retrospective_records(splits.validation, split='validation')
if set(Counter(r.domain.label for r in fit_records)) != set(all_photometry_domain_labels()) or set(Counter(r.domain.label for r in qualification_records)) != set(all_photometry_domain_labels()):
    raise RuntimeError('Complete eligible train/validation inventories must cover all 15 domains.')
inventory = []
for split_name, selected in (('train', fit_records), ('validation', qualification_records)):
    for record in selected:
        identity = fb_cli._classify_variant_a_split_record(record)
        source = Path(record.image_path).resolve(strict=True); relative = source.relative_to(data_root)
        inventory.append({'split': split_name, 'record_identity': str(record.case_id), 'record_identity_sha256': sha256_text(str(record.case_id)), 'subject_group_identity': identity.subject_group_identity, 'domain': record.domain.label, 'relative_source_path': relative.as_posix(), 'source_path_identity_sha256': sha256_text(str(record.image_path)), 'source_bytes': source.stat().st_size, 'source_file_sha256': sha256_file(source)})
inventory.sort(key=lambda x: (x['split'], x['domain'], x['record_identity']))
fit_hash = sha256_json([x for x in inventory if x['split'] == 'train']); validation_hash = sha256_json([x for x in inventory if x['split'] == 'validation'])
if fit_hash != EXPECTED_RETROSPECTIVE_FIT_INVENTORY_SHA256 or validation_hash != FROZEN_RETROSPECTIVE_QUALIFICATION_INVENTORY_SHA256:
    raise RuntimeError('Complete retrospective inventory identity mismatch.')
if FROZEN_SPLIT_V3_JSON.read_bytes() != original_bytes: raise RuntimeError('Original split was modified.')
print(json.dumps({'operational_split': str(OPERATIONAL_SPLIT), 'original_sha256': EXPECTED_SPLIT_V3_SHA256, 'operational_sha256': sha256_file(OPERATIONAL_SPLIT), 'membership': membership, 'recovery': recovery, 'mapping_sha256': mapping_sha, 'fit_count': len(fit_records), 'fit_inventory_sha256': fit_hash, 'validation_count': len(qualification_records), 'validation_inventory_sha256': validation_hash, 'domain_counts_train': dict(sorted(Counter(r.domain.label for r in fit_records).items())), 'domain_counts_validation': dict(sorted(Counter(r.domain.label for r in qualification_records).items())), 'excluded_P_train': fit_excluded, 'excluded_P_validation': qualification_excluded, 'classification_before_array_load': True, 'performance_based_selection': False}, indent=2))


In [ ]:
# Visible, durable command logging. Existing immutable outputs are validated and skipped.
def run_logged(command, log_path, operation):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8', buffering=1) as log:
        log.write(json.dumps({'operation': operation, 'commit': IMPLEMENTATION_REF, 'command': command}) + '\n')
        process = subprocess.Popen(command, cwd=REPO_DIR, env=CLI_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        code = process.wait(); log.write(json.dumps({'operation': operation, 'return_code': code}) + '\n'); log.flush(); os.fsync(log.fileno())
    if code: raise subprocess.CalledProcessError(code, command)
VARIANT_CONFIG = REPO_DIR / 'configs/experiment/stage2_photometry_factorization_a_v1.yaml'
PHOTOMETRY = output_root / 'stage2_photometry_factorization_v1.json'
CONTINUITY = output_root / 'stage2_photometry_continuity_reference_v2.json'
QUALIFICATION = output_root / 'stage2_photometry_variant_a_qualification_v1.json'
if not PHOTOMETRY.exists():
    run_logged([sys.executable, '-m', 'fieldbridge.cli', 'fit-stage2-photometry', '--config', str(VARIANT_CONFIG), '--split-json', str(OPERATIONAL_SPLIT), '--out', str(PHOTOMETRY), '--device', 'cpu', '--log-every', '10'], output_root / 'logs/variant_a_fit.log', 'fit-stage2-photometry')
if not CONTINUITY.exists():
    run_logged([sys.executable, '-m', 'fieldbridge.cli', 'build-stage2-photometry-continuity-reference', '--gate01-result', str(GATE01_RESULT_JSON), '--evaluation-id', 'gate01-external-continuity-only-source-sha256:' + EXPECTED_GATE01_RESULT_SHA256, '--out', str(CONTINUITY)], output_root / 'logs/gate01_continuity.log', 'build-continuity-reference')
if not QUALIFICATION.exists():
    run_logged([sys.executable, '-m', 'fieldbridge.cli', 'audit-stage2-photometry', '--config', str(VARIANT_CONFIG), '--split-json', str(OPERATIONAL_SPLIT), '--artifact', str(PHOTOMETRY), '--vae-config', str(FROZEN_STAGE1_VAE_CONFIG), '--vae-checkpoint', str(FROZEN_VAE_CHECKPOINT), '--continuity-reference', str(CONTINUITY), '--gate01-result', str(GATE01_RESULT_JSON), '--out', str(QUALIFICATION), '--device', 'cuda', '--precision', 'float32', '--log-every', '1'], output_root / 'logs/variant_a_qualification.log', 'audit-stage2-photometry')
qualification = json.loads(QUALIFICATION.read_text())
if qualification.get('canonical_latent_bank_authorized') is not True:
    raise RuntimeError({'Variant_A_qualification_failed': qualification.get('failures')})
if qualification.get('eligibility_proof', {}).get('prospective_accepted_count') != 0:
    raise RuntimeError('Variant A accepted prospective data.')
print(json.dumps({'artifact_file_sha256': sha256_file(PHOTOMETRY), 'artifact_identity': qualification['artifact_sha256'], 'qualification_file_sha256': sha256_file(QUALIFICATION), 'qualification_result_sha256': qualification['result_sha256'], 'authorized': True}, indent=2))


In [ ]:
# Streamed factored-bank filesystem/storage preflight, build, and complete source->N_d->E audit.
CANONICAL_CONFIG = REPO_DIR / 'configs/experiment/stage2_canonical_artifacts_v2.yaml'
BANK_DIR = output_root / 'photometry_factored_latent_bank_v2'
common = ['--config', str(CANONICAL_CONFIG), '--split-json', str(OPERATIONAL_SPLIT), '--photometry-artifact', str(PHOTOMETRY), '--qualification', str(QUALIFICATION), '--vae-config', str(FROZEN_STAGE1_VAE_CONFIG), '--vae-checkpoint', str(FROZEN_VAE_CHECKPOINT)]
run_logged([sys.executable, '-m', 'fieldbridge.cli', 'preflight-photometry-factored-latent-bank', *common, '--out-dir', str(BANK_DIR), '--device', 'cuda'], output_root / 'logs/factored_bank_preflight.log', 'factored-bank-preflight')
if not (BANK_DIR / 'photometry_factored_latent_bank_manifest.json').exists():
    run_logged([sys.executable, '-m', 'fieldbridge.cli', 'build-photometry-factored-latent-bank', *common, '--out-dir', str(BANK_DIR), '--device', 'cuda', '--resume', '--log-every', '1'], output_root / 'logs/factored_bank_build.log', 'factored-bank-build')
run_logged([sys.executable, '-m', 'fieldbridge.cli', 'audit-photometry-factored-latent-bank', *common, '--bank-dir', str(BANK_DIR), '--device', 'cuda', '--log-every', '1'], output_root / 'logs/factored_bank_audit.log', 'factored-bank-audit')
bank_manifest = json.loads((BANK_DIR / 'photometry_factored_latent_bank_manifest.json').read_text())
if bank_manifest['eligibility_proof']['prospective_accepted_count'] != 0 or bank_manifest['record_count'] != len(fit_records) + len(qualification_records):
    raise RuntimeError('Factored bank eligibility/inventory mismatch.')
print(json.dumps({'bank_artifact_sha256': bank_manifest['artifact_sha256'], 'record_count': bank_manifest['record_count'], 'domain_counts': bank_manifest['domain_counts'], 'canonical_persisted': bank_manifest['canonical_stream']['full_canonical_tensor_persisted'], 'support_rule': bank_manifest['operational_support_rule']['contract_version']}, indent=2))


In [ ]:
# Mandatory short full-objective sanity run; weighted auxiliary dominance is a hard failure.
UNIFIED_CONFIG = REPO_DIR / 'configs/experiment/stage2_unified_full_retrospective_v1.yaml'
SANITY_DIR = output_root / 'unified_sanity'
SANITY_CHECKPOINTS = SANITY_DIR / 'checkpoints'; SANITY_HISTORY = SANITY_DIR / 'history.jsonl'
latest_sanity = sorted(SANITY_CHECKPOINTS.glob('stage2_unified_full_step*.pt'))[-1] if SANITY_CHECKPOINTS.exists() and list(SANITY_CHECKPOINTS.glob('stage2_unified_full_step*.pt')) else None
sanity_command = [sys.executable, '-m', 'fieldbridge.cli', 'train-stage2-unified', '--config', str(UNIFIED_CONFIG), '--bank-dir', str(BANK_DIR), '--vae-config', str(FROZEN_STAGE1_VAE_CONFIG), '--vae-checkpoint', str(FROZEN_VAE_CHECKPOINT), '--checkpoint-dir', str(SANITY_CHECKPOINTS), '--history-jsonl', str(SANITY_HISTORY), '--steps', '20', '--sanity-steps', '20', '--device', 'cuda']
if latest_sanity: sanity_command += ['--resume-from', str(latest_sanity)]
run_logged(sanity_command, SANITY_DIR / 'diagnostic.log', 'full-model-sanity')
print({'sanity_history': str(SANITY_HISTORY), 'latest_checkpoint': str(sorted(SANITY_CHECKPOINTS.glob('stage2_unified_full_step*.pt'))[-1])})


In [ ]:
# The only long-run authorization. Full model runs first, then backward ablations, then final evaluation.
RUN_LONG_FULL_AND_BACKWARD_ABLATIONS = False
if RUN_LONG_FULL_AND_BACKWARD_ABLATIONS is not True:
    raise PermissionError('Set the single long-run authorization flag to True to launch/resume training and evaluation.')
import yaml
base_config = yaml.safe_load(UNIFIED_CONFIG.read_text())
variants = {
 'full': {},
 'no_graph': {'graph': 0.0},
 'no_anatomy_graph': {'anatomy': 0.0, 'graph': 0.0},
 'no_adversarial_domain': {'adversarial': 0.0, 'domain': 0.0},
 'sb_identity_only': {'anatomy': 0.0, 'graph': 0.0, 'adversarial': 0.0, 'domain': 0.0},
 'sb_only': {'identity': 0.0, 'anatomy': 0.0, 'graph': 0.0, 'adversarial': 0.0, 'domain': 0.0},
}
latest_by_variant = {}
for variant, overrides in variants.items():
    cfg = copy.deepcopy(base_config); cfg['training']['variant'] = variant; cfg['training']['loss_weights'].update(overrides); cfg['training']['sanity']['steps'] = 0
    run_dir = output_root / 'unified_training' / variant; checkpoint_dir = run_dir / 'checkpoints'; history = run_dir / 'history.jsonl'; config_path = run_dir / 'resolved_config.json'; run_dir.mkdir(parents=True, exist_ok=True)
    if config_path.exists() and json.loads(config_path.read_text()) != cfg: raise RuntimeError(f'Variant config changed for exact resume: {variant}')
    if not config_path.exists(): write_json_atomic(config_path, cfg)
    candidates = sorted(checkpoint_dir.glob(f'stage2_unified_{variant}_step*.pt')) if checkpoint_dir.exists() else []
    command = [sys.executable, '-m', 'fieldbridge.cli', 'train-stage2-unified', '--config', str(config_path), '--bank-dir', str(BANK_DIR), '--vae-config', str(FROZEN_STAGE1_VAE_CONFIG), '--vae-checkpoint', str(FROZEN_VAE_CHECKPOINT), '--checkpoint-dir', str(checkpoint_dir), '--history-jsonl', str(history), '--device', 'cuda']
    if candidates: command += ['--resume-from', str(candidates[-1])]
    run_logged(command, run_dir / 'diagnostic.log', f'train-{variant}')
    latest_by_variant[variant] = sorted(checkpoint_dir.glob(f'stage2_unified_{variant}_step*.pt'))[-1]
EVALUATION_DIR = output_root / 'unified_retrospective_evaluation'
eval_command = [sys.executable, '-m', 'fieldbridge.cli', 'eval-stage2-unified', '--config', str(UNIFIED_CONFIG), '--bank-dir', str(BANK_DIR), '--checkpoint', str(latest_by_variant['full']), '--sb-only-checkpoint', str(latest_by_variant['sb_only']), '--vae-config', str(FROZEN_STAGE1_VAE_CONFIG), '--vae-checkpoint', str(FROZEN_VAE_CHECKPOINT), '--photometry-artifact', str(PHOTOMETRY), '--paired-manifest', str(PAIRED_R_VALIDATION_MANIFEST), '--baseline-predictions', str(BASELINE_PREDICTIONS_MANIFEST), '--out', str(EVALUATION_DIR), '--device', 'cuda', '--integration-steps', '20', '--solver', 'heun', '--resume']
run_logged(eval_command, output_root / 'logs/unified_evaluation.log', 'unified-retrospective-evaluation')
result = json.loads((EVALUATION_DIR / 'result.json').read_text())
if result['prospective_records_loaded'] != 0 or result['descriptor_coupling_used'] is not False:
    raise RuntimeError('Final evaluation crossed an authorization boundary.')
print(json.dumps({'result_sha256': result['result_sha256'], 'case_count': result['case_count'], 'methods': result['methods'], 'reductions': result['reductions'], 'montages': str(EVALUATION_DIR / 'montages')}, indent=2))


In [ ]:
# Seal a compact paper-review index without touching repository files.
evidence = {'contract': 'stage2-unified-colab-evidence-index-v1', 'implementation_commit': IMPLEMENTATION_REF, 'sealed_inputs': {label: {'path': str(path), 'file_sha256': sha256_file(path)} for label, (path, _) in file_inputs.items()}, 'operational_split_sha256': sha256_file(OPERATIONAL_SPLIT), 'photometry_artifact_sha256': sha256_file(PHOTOMETRY), 'qualification_sha256': sha256_file(QUALIFICATION), 'bank_manifest_sha256': sha256_file(BANK_DIR / 'photometry_factored_latent_bank_manifest.json'), 'full_checkpoint_sha256': sha256_file(latest_by_variant['full']), 'sb_only_checkpoint_sha256': sha256_file(latest_by_variant['sb_only']), 'evaluation_result_sha256': sha256_file(EVALUATION_DIR / 'result.json'), 'prospective_execution': False, 'descriptor_coupling': False}
evidence['evidence_sha256'] = sha256_json(evidence)
evidence_path = output_root / 'stage2_unified_evidence_index.json'
if evidence_path.exists():
    if json.loads(evidence_path.read_text()) != evidence: raise RuntimeError('Existing evidence index mismatch.')
else: write_json_atomic(evidence_path, evidence)
if FROZEN_SPLIT_V3_JSON.read_bytes() != original_bytes or git_text('status', '--porcelain'):
    raise RuntimeError('Immutable split or detached checkout changed.')
print(json.dumps(evidence, indent=2))
